In [ ]:
! pip install datasets
! pip uninstall -y torchvision -q

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling
import torch

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### SFT

In [4]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
text = "Hey! This is Mehak!"
tokens = tokenizer(text)
print(tokens)

{'input_ids': [10814, 0, 770, 318, 37070, 461, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [6]:
ds = load_dataset("stanfordnlp/sst2")

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [7]:
ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [8]:
ds_train, ds_val = ds["train"], ds["validation"]
ds_train

Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 67349
})

In [9]:
ds_train[6:9]

{'idx': [6, 7, 8],
 'sentence': ['demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop . ',
  'of saucy ',
  "a depressed fifteen-year-old 's suicidal poetry "],
 'label': [1, 1, 0]}

In [10]:
# tokenize
def tokenize(batch):
  return tokenizer(batch["sentence"])

map_kwargs = {
    "batched": True,
    "batch_size": 512,
    "remove_columns": ["idx", "sentence", "label"]
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [11]:
tokenized_dataset_train[0]

{'input_ids': [24717, 649, 3200, 507, 422, 262, 21694, 4991, 220],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x["input_ids"]) > 5)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x["input_ids"]) > 5)

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/872 [00:00<?, ? examples/s]

In [13]:
print(len(tokenized_dataset_train), len(tokenized_dataset_val))

49401 867


In [14]:
tokenized_dataset_train.set_format(type="torch")
tokenized_dataset_val.set_format(type="torch")

In [15]:
tokenized_dataset_train[0]

{'input_ids': tensor([24717,   649,  3200,   507,   422,   262, 21694,  4991,   220]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1])}

In [16]:
tokenized_dataset_train[:5]

{'input_ids': [tensor([24717,   649,  3200,   507,   422,   262, 21694,  4991,   220]),
  tensor([ 3642,  1299,   645, 20868,   837,   691,  2248,  1850,   308,  3775,
            220]),
  tensor([ 5562, 10408,   663,  3435,   290, 48556,  1223,  2138,  4950,   546,
           1692,  3450,   220]),
  tensor([ 2787,  1299, 15950, 11378,   284,  3520,   262,   976,  3690,   220]),
  tensor([  261,   262,  5290, 15827,    12,  1659,    12,  1169,    12,  1008,
           9310, 35478, 20954,   262, 28303,   714, 47478,   469,   510,   220])],
 'attention_mask': [tensor([1, 1, 1, 1, 1, 1, 1, 1, 1]),
  tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
  tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
  tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
  tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])]}

In [17]:
print(tokenizer.pad_token)
print(tokenizer.eos_token)
tokenizer.pad_token = tokenizer.eos_token

None
<|endoftext|>


In [18]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

dataloader_params = {
    "batch_size": 32,
    "collate_fn": data_collator
}

train_dataloader = DataLoader(tokenized_dataset_train, **dataloader_params)
val_dataloader = DataLoader(tokenized_dataset_val, **dataloader_params)

In [19]:
len(train_dataloader) * 32

49408

In [20]:
batch = next(iter(train_dataloader))
print(batch.keys())

KeysView({'input_ids': tensor([[24717,   649,  3200,  ..., 50256, 50256, 50256],
        [ 3642,  1299,   645,  ..., 50256, 50256, 50256],
        [ 5562, 10408,   663,  ..., 50256, 50256, 50256],
        ...,
        [  672, 35260,   284,  ..., 50256, 50256, 50256],
        [ 1169,  2104,   966,  ..., 50256, 50256, 50256],
        [  292,   484,  1282,  ..., 50256, 50256, 50256]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[24717,   649,  3200,  ...,  -100,  -100,  -100],
        [ 3642,  1299,   645,  ...,  -100,  -100,  -100],
        [ 5562, 10408,   663,  ...,  -100,  -100,  -100],
        ...,
        [  672, 35260,   284,  ...,  -100,  -100,  -100],
        [ 1169,  2104,   966,  ...,  -100,  -100,  -100],
        [  292,   484,  1282,  ...,  -100,  -100,  -100]])})


In [21]:
print(batch["input_ids"].shape) # (B, T)
print(batch["input_ids"][0])
print(batch["labels"][0]) # -100: padding token
print(batch["attention_mask"][0])

torch.Size([32, 35])
tensor([24717,   649,  3200,   507,   422,   262, 21694,  4991,   220, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256])
tensor([24717,   649,  3200,   507,   422,   262, 21694,  4991,   220,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [22]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 1

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [24]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_dataloader):
        batch = batch.to(device)

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        print(f"Epoch {epoch+1} | Step {step+1}/{len(train_dataloader)} | Loss = {loss.item():.4f}")

    avg_loss = total_loss / len(train_dataloader)
    print(f"\nEpoch {epoch+1} Average Loss = {avg_loss:.4f}")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1 | Step 1/1544 | Loss = 5.8525
Epoch 1 | Step 2/1544 | Loss = 5.6948
Epoch 1 | Step 3/1544 | Loss = 5.1838
Epoch 1 | Step 4/1544 | Loss = 5.2326
Epoch 1 | Step 5/1544 | Loss = 5.3562
Epoch 1 | Step 6/1544 | Loss = 5.3954
Epoch 1 | Step 7/1544 | Loss = 5.1112
Epoch 1 | Step 8/1544 | Loss = 4.9385
Epoch 1 | Step 9/1544 | Loss = 4.8394
Epoch 1 | Step 10/1544 | Loss = 5.0041
Epoch 1 | Step 11/1544 | Loss = 4.8588
Epoch 1 | Step 12/1544 | Loss = 4.7501
Epoch 1 | Step 13/1544 | Loss = 4.8499
Epoch 1 | Step 14/1544 | Loss = 4.9300
Epoch 1 | Step 15/1544 | Loss = 4.4851
Epoch 1 | Step 16/1544 | Loss = 4.4867
Epoch 1 | Step 17/1544 | Loss = 4.5657
Epoch 1 | Step 18/1544 | Loss = 4.5714
Epoch 1 | Step 19/1544 | Loss = 4.6933
Epoch 1 | Step 20/1544 | Loss = 4.5931
Epoch 1 | Step 21/1544 | Loss = 4.6553
Epoch 1 | Step 22/1544 | Loss = 4.4965
Epoch 1 | Step 23/1544 | Loss = 4.4175
Epoch 1 | Step 24/1544 | Loss = 4.5770
Epoch 1 | Step 25/1544 | Loss = 4.4719
Epoch 1 | Step 26/1544 | Loss = 4.

In [27]:
save_path = "/content/drive/MyDrive/sft_model"
model.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]